# 1. Introduction
In medical AI, high accuracy is not enough. we need **Explainability (XAI)** to trust the model's diagnosis.
This notebook focuses on **interpreting the model** using SHAP values, rather than just chasing the leaderboard score.

<div class="alert alert-block alert-info">
<b>🎯 Goal:</b> To visualize "Why" the model predicts heart disease, identifying linear and non-linear risk factors.
</div>

# 2. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Load Data
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")
sample_sub = pd.read_csv("/kaggle/input/playground-series-s6e2/sample_submission.csv")

# Target Encoding (Presence=1, Absence=0)
train['Heart Disease'] = train['Heart Disease'].apply(lambda x: 1 if x == 'Presence' else 0)

# Prepare X and y
X = train.drop(['id', 'Heart Disease'], axis=1)
y = train['Heart Disease']
X_test = test.drop(['id'], axis=1)

# Split Data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Setup Complete!")

# 3. Modeling (LightGBM)

In [ ]:
# Best Params (Based on Optuna tuning)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

# Train Model
model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print(f"Validation AUC: {model.best_score_['valid_0']['auc']:.4f}")

# 4. SHAP Visualization

In [ ]:
# Calculate SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_val)

# Summary Plot
plt.figure(figsize=(10, 8))
plt.title("Feature Importance: What drives Heart Disease?", fontsize=16)
shap.summary_plot(shap_values, X_val, show=False) 
plt.show()

# 5. Deep Dive: Decoding the Black Box
Using SHAP dependence plots, we found three types of risk behaviors.

### Type A: The "Switches" (Binary Risks)
Features like **Chest Pain Type** and **Thallium** act as decisive switches.
There is a clear gap between low risk (blue) and high risk (red).

In [ ]:
# Chest Pain Type
shap.dependence_plot("Chest pain type", shap_values, X_val, interaction_index=None)

<div class="alert alert-block alert-danger">
<b>💡 Insight:</b> "Asymptomatic" (Type 4) pain is the most dangerous. The model captures the "Silent Killer" phenomenon perfectly.
</div>

### Type B: The "Non-Linears" (Threshold Risks)
**ST depression** shows a complex pattern. It's not just "higher is worse".
Risk spikes only after exceeding a certain threshold.

In [ ]:
# ST depression
shap.dependence_plot("ST depression", shap_values, X_val, interaction_index=None)

### Type C: The "Saturation" (Plateau Effect)
For **Number of vessels fluro**, risk increases from 0 to 1 to 2, but then **flattens out (plateaus)**.
This suggests a **"Saturation Effect"**: having multiple blocked vessels is already critical, so the model sees little difference between 2 and 3 vessels. Both are maximum risk.

In [ ]:
# Number of vessels fluro
shap.dependence_plot("Number of vessels fluro", shap_values, X_val, interaction_index=None)

### Type D: The "Paradox" (Counter-Intuitive Results)
Surprisingly, **Cholesterol** shows little correlation with risk in this model.
<div class="alert alert-block alert-warning">
<b>⚠️ The Cholesterol Paradox:</b>
Unlike popular belief, Cholesterol does not show a clear linear risk pattern.
This is likely due to the <b>"Medication Effect"</b>. High-risk patients are often already taking medication (statins) to lower their levels.
Thus, the model might see "Low Cholesterol = High Risk Patient", creating a paradox.
</div>

In [ ]:
# Cholesterol
shap.dependence_plot("Cholesterol", shap_values, X_val, interaction_index=None)

# 6. Conclusion
By using SHAP, we confirmed that the LightGBM model is not just memorizing data, but learning **medically valid patterns**:

1.  **Switches:** Features like **Chest Pain Type** act as binary flags (Normal vs Abnormal).
2.  **Thresholds:** **ST depression** shows a J-curve risk pattern (safe until a tipping point).
3.  **Saturation:** **Vessels** risk increases initially but **plateaus** at higher counts (Diminishing Returns).
4.  **Paradox:** The model reveals hidden factors like the **Medication Effect** in Cholesterol.

If you found this notebook helpful for understanding Medical XAI, **please Upvote!** ⬆️

In [ ]:
# Create Submission
# Predict probabilities (using the same features)
y_pred_prob = model.predict_proba(X_test)[:, 1]

# Create DataFrame
submission = sample_sub.copy()
submission['Heart Disease'] = y_pred_prob

# Save
submission.to_csv("submission.csv", index=False)
print("Submission saved successfully!")

# Check format
submission.head()